# Retriever Eval

In [11]:
import torch
from transformers import BitsAndBytesConfig,AutoTokenizer,AutoModelForCausalLM
import faiss
import pickle
from prepare_data import lemmatize_en
from eval_pipline import complete_eval_pipline
from metrics import compute_agg_metrics

In [12]:

name_model="Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [14]:
tokenizer=AutoTokenizer.from_pretrained(name_model)
model = AutoModelForCausalLM.from_pretrained(
    name_model,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map='auto',
    attn_implementation="sdpa"

)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## BGE-M3

In [30]:
data_oracle_with_metrics,result_oracle=complete_eval_pipline(is_oracle=True,
                                    is_generate_answers=True,
                                      path_to_eval_set='/kaggle/input/datasets/vorange/labeled-eval-set-rag-system-anime/eval_set_chunks_sampled_labeled.csv',
                                    path_to_data_with_answers_oracle='/kaggle/input/datasets/vorange/labeled-eval-set-rag-system-anime/eval_set_chunks_sampled_labeled.csv',
                                    is_simple_generation_metrics=True,
                                    is_generation_metrics=True,
                                    is_time_metrics = True,
                                    list_columns_time =['generation_time','e2e_latency'],
                                    is_abstention_metrics =True,
                                    queries_column='question',
                                    answers_column='llm_answer',
                                    chunks_column='relevant_chunks',
                                    ground_truth_column='ground_truth_text',
                                    generation_source='local',
                                    tokenizer=tokenizer,
                                    local_generation_model=model,
                                    name_evaluation_model='llama-3.1-8b-instant',
                                                      )

  0%|          | 0/101 [00:00<?, ?it/s]

/tmp/ipykernel_55/502526891.py:122: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  return LangchainLLMWrapper(groq_llm)
/tmp/ipykernel_55/502526891.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), per_group)))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 3.71 seconds, 27.23 sentences/sec
Exact Match: 0.000
BERTScore F1: 0.847
BERTScore Precision: 0.859
BERTScore Recall: 0.837
generation_time mean: 4.187456873950513
generation_time median: 2.7907784239996545
generation_time p95 : 8.412893579999945
e2e_latency mean: 4.1874824209505075
e2e_latency median: 2.790808309000113
e2e_latency p95 : 8.412921539999843
Точность правильных отказов: 100.0%
Точность ложных отказов: 4.3%


In [42]:
data_oracle_with_metrics.to_csv('data_oracle_with_metrics.csv')

In [40]:
compute_agg_metrics(data_oracle_with_metrics,
                is_generation_metrics=True,
        aggregation_generation_columns = ['language','answer_type'],
        is_simple_generation_metrics = True,
        simple_aggregation_generation_columns = ['language','answer_type'],
        simple_generation_metrics_columns=['bertscore_f1'],
        is_retriever_metrics=False,
        aggregation_retriever_columns=['language','answer_type'], 
        is_abstention_metrics = True,
        aggregation_abstention_columns =['language','answer_type'],
)

Generation:
           faithfulness  answer_correctness  answer_relevancy
language                                                    
en            0.745238            0.502855          0.260611
ru                 NaN                 NaN          0.000000 

Generation:
               faithfulness  answer_correctness  answer_relevancy
answer_type                                                     
abstention             NaN                 NaN               NaN
descriptive       0.500000            0.349498         -0.033187
list                   NaN                 NaN               NaN
short_entity      0.752451            0.507647          0.261462 

Simple generation:
           bertscore_f1
language              
en            0.847118
ru            0.847210 

Simple generation:
               bertscore_f1
answer_type               
abstention        0.761899
descriptive       0.847758
list              0.877344
short_entity      0.855447 

Abstained metrics:
 language
en    0.1

In [16]:
faiss_index_bge=faiss.read_index('bm3/faiss/all_fandom_bge-m3.faiss')
data_bge_m3_with_metrics,result_bge_m3=complete_eval_pipline(is_oracle=False,
                        is_generate_answers=True,
                        path_to_eval_set_with_chunks='eval_set_chunks_sampled_labeled.csv',
                        retriever_model='bge-m3',
                        is_simple_generation_metrics=True,
                        is_time_metrics = True,
                        list_columns_time =['generation_time','e2e_latency'],
                        is_abstention_metrics =True,
                        queries_column='question',
                        answers_column='llm_answer',
                        chunks_column='chunks_from_retrieval',
                        relevant_chunks_column = 'llm_relevant_chunks_text',
                        ground_truth_column='ground_truth_text',
                        retriever_type='faiss',
                        faiss_index=faiss_index_bge,
                        top_k_chunks=5,
                        is_alone_retriever=True,
                        generation_source='local',
                        tokenizer=tokenizer,
                        local_generation_model=model,
                        name_evaluation_model='llama-3.1-8b-instant',
                        threshold = None,
                        is_retrieval_metrics=True,
                        top_k_recall=[1,3,5],
                        top_k_precision=[1,3,5],
                        top_k_hit=[1,3,5],
                        top_k_nDCG=[1,3,5],
                        is_context_precision=True

                                                      )

  0%|          | 0/101 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 4.03 seconds, 25.07 sentences/sec
Exact Match: 0.000
BERTScore F1: 0.832
BERTScore Precision: 0.836
BERTScore Recall: 0.830
generation_time mean: 10.624611921049507
generation_time median: 7.479894152000043
generation_time p95 : 32.565333021000015
e2e_latency mean: 10.662911825544565
e2e_latency median: 7.516486100000066
e2e_latency p95 : 32.59896783299996
top_1_recall: 0.1945247524752475
top_3_recall: 0.3845940594059406
top_5_recall: 0.45037623762376244
top_1_precision: 0.38613861386138615
top_3_precision: 0.27052475247524754
top_5_precision: 0.21188118811881188
top_1_hit: 0.38613861386138615
top_3_hit: 0.6039603960396039
top_5_hit: 0.6534653465346535
top_1_nDCG: 0.38613861386138615
top_3_nDCG: 0.4043960396039604
top_5_nDCG: 0.4153960396039604
context_precision: 0.46849504950495047
Точность правильных отказов: 44.4%
Точность ложных отказов: 5.4%


In [25]:
retriever_metrics_columns=[
    "top_1_recall",
    "top_3_recall",
    "top_5_recall",

    "top_1_precision",
    "top_3_precision",
    "top_5_precision",

    "top_1_hit",
    "top_3_hit",
    "top_5_hit",

    "top_1_nDCG",
    "top_3_nDCG",
    "top_5_nDCG",

    "context_precision",
]

In [54]:
data_bge_m3_with_metrics.to_csv('data_bge_m3_with_metrics.csv')

In [18]:
compute_agg_metrics(data_bge_m3_with_metrics,
        is_simple_generation_metrics = True,
        simple_aggregation_generation_columns = ['language','answer_type'],
        simple_generation_metrics_columns=['bertscore_f1'],
        is_retriever_metrics=True,
        aggregation_retriever_columns=['language','answer_type'], 
        retriever_metrics_columns=retriever_metrics_columns,
        is_abstention_metrics = True,
        aggregation_abstention_columns =['language','answer_type'],
)

Simple generation:
           bertscore_f1
language              
en            0.830248
ru            0.833152 

Simple generation:
               bertscore_f1
answer_type               
abstention        0.770463
descriptive       0.842768
list              0.835466
short_entity      0.837462 

Retriever metrics:
           top_1_recall  top_3_recall  top_5_recall  top_1_precision  \
language                                                              
en            0.185840      0.382780      0.463260         0.400000   
ru            0.203039      0.386373      0.437745         0.372549   

          top_3_precision  top_5_precision  top_1_hit  top_3_hit  top_5_hit  \
language                                                                      
en               0.279900         0.220000   0.400000   0.620000   0.680000   
ru               0.261333         0.203922   0.372549   0.588235   0.627451   

          top_1_nDCG  top_3_nDCG  top_5_nDCG  context_precision  
language      

In [19]:
faiss_index_e5=faiss.read_index('multi-e5/faiss/all_fandom_e5-large.faiss')
data_e5_with_metrics,result_e5=complete_eval_pipline(is_oracle=False,
                        is_generate_answers=True,
                        path_to_eval_set_with_chunks='eval_set_chunks_sampled_labeled.csv',
                        retriever_model='multi-e5',
                        is_simple_generation_metrics=True,
                        is_time_metrics = True,
                        list_columns_time =['generation_time','e2e_latency'],
                        is_abstention_metrics =True,
                        queries_column='question',
                        answers_column='llm_answer',
                        chunks_column='chunks_from_retrieval',
                        relevant_chunks_column = 'llm_relevant_chunks_text',
                        ground_truth_column='ground_truth_text',
                        retriever_type='faiss',
                        faiss_index=faiss_index_e5,
                        top_k_chunks=5,
                        is_alone_retriever=True,
                        generation_source='local',
                        tokenizer=tokenizer,
                        local_generation_model=model,
                        name_evaluation_model='llama-3.1-8b-instant',
                        threshold = None,
                        is_retrieval_metrics=True,
                        top_k_recall=[1,3,5],
                        top_k_precision=[1,3,5],
                        top_k_hit=[1,3,5],
                        top_k_nDCG=[1,3,5],
                        is_context_precision=True

                                                      )

  0%|          | 0/101 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 2.42 seconds, 41.66 sentences/sec
Exact Match: 0.000
BERTScore F1: 0.840
BERTScore Precision: 0.849
BERTScore Recall: 0.833
generation_time mean: 8.236453581940603
generation_time median: 6.664257280999891
generation_time p95 : 19.014327515999867
e2e_latency mean: 8.258463177168323
e2e_latency median: 6.686246382000036
e2e_latency p95 : 19.037514271999953
top_1_recall: 0.17743564356435643
top_3_recall: 0.38509900990099005
top_5_recall: 0.45415841584158423
top_1_precision: 0.32673267326732675
top_3_precision: 0.24409900990099012
top_5_precision: 0.19009900990099005
top_1_hit: 0.32673267326732675
top_3_hit: 0.5841584158415841
top_5_hit: 0.6633663366336634
top_1_nDCG: 0.32673267326732675
top_3_nDCG: 0.37886138613861386
top_5_nDCG: 0.3957326732673267
context_precision: 0.45213861386138615
Точность правильных отказов: 55.6%
Точность ложных отказов: 8.7%


In [55]:
data_e5_with_metrics.to_csv('data_e5_with_metrics.csv')

In [20]:
compute_agg_metrics(data_e5_with_metrics,
        is_simple_generation_metrics = True,
        simple_aggregation_generation_columns = ['language','answer_type'],
        simple_generation_metrics_columns=['bertscore_f1'],
        is_retriever_metrics=True,
        aggregation_retriever_columns=['language','answer_type'], 
        retriever_metrics_columns=retriever_metrics_columns,
        is_abstention_metrics = True,
        aggregation_abstention_columns =['language','answer_type'],
)

Simple generation:
           bertscore_f1
language              
en            0.842716
ru            0.837739 

Simple generation:
               bertscore_f1
answer_type               
abstention        0.777390
descriptive       0.863592
list              0.883641
short_entity      0.844456 

Retriever metrics:
           top_1_recall  top_3_recall  top_5_recall  top_1_precision  \
language                                                              
en            0.216860      0.403220      0.499800          0.38000   
ru            0.138784      0.367333      0.409412          0.27451   

          top_3_precision  top_5_precision  top_1_hit  top_3_hit  top_5_hit  \
language                                                                      
en               0.266540         0.216000    0.38000    0.62000   0.740000   
ru               0.222098         0.164706    0.27451    0.54902   0.588235   

          top_1_nDCG  top_3_nDCG  top_5_nDCG  context_precision  
language      

In [19]:
with open("chunks/all_fandom_chunks.pkl", "rb") as f:
                    all_chunks = pickle.load(f)

tokenized_chunks = [lemmatize_en(c['text']) for c in all_chunks]

In [22]:

data_bm25_with_metrics,result_bm25=complete_eval_pipline(is_oracle=False,
                        is_generate_answers=True,
                        path_to_eval_set_with_chunks='eval_set_chunks_sampled_labeled.csv',
                        retriever_model='multi-e5',
                        is_simple_generation_metrics=True,
                        is_time_metrics = True,
                        list_columns_time =['generation_time','e2e_latency'],
                        is_abstention_metrics =True,
                        queries_column='question',
                        answers_column='llm_answer',
                        chunks_column='chunks_from_retrieval',
                        relevant_chunks_column = 'llm_relevant_chunks_text',
                        ground_truth_column='ground_truth_text',
                        retriever_type='bm25',
                        tokenized_chunks=tokenized_chunks,
                        faiss_index=None,
                        top_k_chunks=5,
                        is_alone_retriever=True,
                        generation_source='local',
                        tokenizer=tokenizer,
                        local_generation_model=model,
                        name_evaluation_model='llama-3.1-8b-instant',
                        threshold = None,
                        is_retrieval_metrics=True,
                        top_k_recall=[1,3,5],
                        top_k_precision=[1,3,5],
                        top_k_hit=[1,3,5],
                        top_k_nDCG=[1,3,5],
                        is_context_precision=True

                                                      )

  0%|          | 0/101 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 1.71 seconds, 59.13 sentences/sec
Exact Match: 0.000
BERTScore F1: 0.831
BERTScore Precision: 0.842
BERTScore Recall: 0.822
generation_time mean: 7.021006244999996
generation_time median: 6.273431304000042
generation_time p95 : 10.979025036000053
e2e_latency mean: 8.063918982712865
e2e_latency median: 7.317445934000034
e2e_latency p95 : 12.024890264000078
top_1_recall: 0.13301980198019803
top_3_recall: 0.2446930693069307
top_5_recall: 0.2729207920792079
top_1_precision: 0.27722772277227725
top_3_precision: 0.18147524752475247
top_5_precision: 0.12871287128712872
top_1_hit: 0.27722772277227725
top_3_hit: 0.37623762376237624
top_5_hit: 0.39603960396039606
top_1_nDCG: 0.27722772277227725
top_3_nDCG: 0.2668613861386139
top_5_nDCG: 0.2641782178217822
context_precision: 0.3190198019801981
Точность правильных отказов: 22.2%
Точность ложных отказов: 34.8%


In [56]:
data_bm25_with_metrics.to_csv('data_bm25_with_metrics.csv')

In [26]:
compute_agg_metrics(data_bm25_with_metrics,
        is_simple_generation_metrics = True,
        simple_aggregation_generation_columns = ['language','answer_type'],
        simple_generation_metrics_columns=['bertscore_f1'],
        is_retriever_metrics=True,
        aggregation_retriever_columns=['language','answer_type'], 
        retriever_metrics_columns=retriever_metrics_columns,
        is_abstention_metrics = True,
        aggregation_abstention_columns =['language','answer_type'],
)

Simple generation:
           bertscore_f1
language              
en            0.838856
ru            0.823589 

Simple generation:
               bertscore_f1
answer_type               
abstention        0.769295
descriptive       0.835323
list              0.869360
short_entity      0.836551 

Retriever metrics:
           top_1_recall  top_3_recall  top_5_recall  top_1_precision  \
language                                                              
en              0.2687       0.49428        0.5513             0.56   
ru              0.0000       0.00000        0.0000             0.00   

          top_3_precision  top_5_precision  top_1_hit  top_3_hit  top_5_hit  \
language                                                                      
en                0.36658             0.26       0.56       0.76        0.8   
ru                0.00000             0.00       0.00       0.00        0.0   

          top_1_nDCG  top_3_nDCG  top_5_nDCG  context_precision  
language      

In [22]:

data_hybrid_bm_bge_with_metrics,result_hybrid_bm_bge=complete_eval_pipline(is_oracle=False,
                        is_generate_answers=True,
                        path_to_eval_set_with_chunks='eval_set_chunks_sampled_labeled.csv',
                        retriever_model='bge-m3',
                        is_hybrid=True,
                        is_simple_generation_metrics=True,
                        is_time_metrics = True,
                        list_columns_time =['generation_time','e2e_latency'],
                        is_abstention_metrics =True,
                        queries_column='question',
                        answers_column='llm_answer',
                        chunks_column='chunks_from_retrieval',
                        relevant_chunks_column = 'llm_relevant_chunks_text',
                        ground_truth_column='ground_truth_text',
                        retriever_type='faiss',
                        faiss_index=faiss_index_bge,
                        top_k_chunks=5,
                        is_alone_retriever=True,
                        generation_source='local',
                        tokenizer=tokenizer,
                        local_generation_model=model,
                        name_evaluation_model='llama-3.1-8b-instant',
                        threshold = None,
                        is_retrieval_metrics=True,
                        top_k_recall=[1,3,5],
                        top_k_precision=[1,3,5],
                        top_k_hit=[1,3,5],
                        top_k_nDCG=[1,3,5],
                        is_context_precision=True

                                                      )

  0%|          | 0/101 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 2.21 seconds, 45.65 sentences/sec
Exact Match: 0.000
BERTScore F1: 0.842
BERTScore Precision: 0.851
BERTScore Recall: 0.836
generation_time mean: 8.511437418910916
generation_time median: 6.525643818999924
generation_time p95 : 21.725515580999854
e2e_latency mean: 10.637736662950468
e2e_latency median: 8.788505093000367
e2e_latency p95 : 24.00279188500008
top_1_recall: 0.2135049504950495
top_3_recall: 0.3813960396039604
top_5_recall: 0.45654455445544556
top_1_precision: 0.40594059405940597
top_3_precision: 0.263960396039604
top_5_precision: 0.19801980198019803
top_1_hit: 0.40594059405940597
top_3_hit: 0.5742574257425742
top_5_hit: 0.6831683168316832
top_1_nDCG: 0.40594059405940597
top_3_nDCG: 0.40126732673267324
top_5_nDCG: 0.41412871287128705
context_precision: 0.4782574257425743
Точность правильных отказов: 44.4%
Точность ложных отказов: 4.3%


In [57]:
data_hybrid_bm_bge_with_metrics.to_csv('data_hybrid_bm_bge_with_metrics.csv')

In [24]:
compute_agg_metrics(data_hybrid_bm_bge_with_metrics,
        is_simple_generation_metrics = True,
        simple_aggregation_generation_columns = ['language','answer_type'],
        simple_generation_metrics_columns=['bertscore_f1'],
        is_retriever_metrics=True,
        aggregation_retriever_columns=['language','answer_type'], 
        retriever_metrics_columns=retriever_metrics_columns,
        is_abstention_metrics = True,
        aggregation_abstention_columns =['language','answer_type'],
)

Simple generation:
           bertscore_f1
language              
en            0.844707
ru            0.840332 

Simple generation:
               bertscore_f1
answer_type               
abstention        0.772615
descriptive       0.852563
list              0.960599
short_entity      0.846527 

Retriever metrics:
           top_1_recall  top_3_recall  top_5_recall  top_1_precision  \
language                                                              
en            0.224180       0.46288      0.528120         0.440000   
ru            0.203039       0.30151      0.386373         0.372549   

          top_3_precision  top_5_precision  top_1_hit  top_3_hit  top_5_hit  \
language                                                                      
en               0.326580         0.240000   0.440000    0.70000   0.780000   
ru               0.202569         0.156863   0.372549    0.45098   0.588235   

          top_1_nDCG  top_3_nDCG  top_5_nDCG  context_precision  
language      

In [41]:

data_hybrid_bm_bge_with_reranker_with_metrics,result_hybrid_bm_bge_with_reranker=complete_eval_pipline(is_oracle=False,
                        is_generate_answers=True,
                        path_to_eval_set_with_chunks='/kaggle/input/datasets/vorange/labeled-eval-set-rag-system-anime/eval_set_chunks_sampled_labeled.csv',
                        retriever_model='bge-m3',
                        is_hybrid=True,
                        use_reranker=True,
                        top_k_for_reranker=5,
                        is_simple_generation_metrics=True,
                        is_time_metrics = True,
                        list_columns_time =['generation_time','e2e_latency'],
                        is_abstention_metrics =True,
                        queries_column='question',
                        answers_column='llm_answer',
                        chunks_column='chunks_from_retrieval',
                        relevant_chunks_column = 'llm_relevant_chunks_text',
                        ground_truth_column='ground_truth_text',
                        retriever_type='faiss',
                        faiss_index=faiss_index_bge,
                        top_k_chunks=20,
                        is_alone_retriever=True,
                        generation_source='local',
                        tokenizer=tokenizer,
                        local_generation_model=model,
                        name_evaluation_model='llama-3.1-8b-instant',
                        threshold = None,
                        is_retrieval_metrics=True,
                        top_k_recall=[1,3,5],
                        top_k_precision=[1,3,5],
                        top_k_hit=[1,3,5],
                        top_k_nDCG=[1,3,5],
                        is_context_precision=True

                                                      )

  0%|          | 0/101 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 2.27 seconds, 44.54 sentences/sec
Exact Match: 0.000
BERTScore F1: 0.837
BERTScore Precision: 0.845
BERTScore Recall: 0.830
generation_time mean: 9.000204190782249
generation_time median: 7.36803719100044
generation_time p95 : 19.248983403999773
e2e_latency mean: 11.715450310871338
e2e_latency median: 10.055902805000187
e2e_latency p95 : 22.121812506000424
top_1_recall: 0.3185346534653465
top_3_recall: 0.4867128712871287
top_5_recall: 0.5437425742574258
top_1_precision: 0.594059405940594
top_3_precision: 0.34315841584158424
top_5_precision: 0.24950495049504953
top_1_hit: 0.594059405940594
top_3_hit: 0.7128712871287128
top_5_hit: 0.7425742574257426
top_1_nDCG: 0.594059405940594
top_3_nDCG: 0.5498712871287129
top_5_nDCG: 0.546970297029703
context_precision: 0.6297326732673267
Точность правильных отказов: 77.8%
Точность ложных отказов: 9.8%


In [58]:
data_hybrid_bm_bge_with_reranker_with_metrics.to_csv('data_hybrid_bm_bge_with_reranker_with_metrics.csv')

In [42]:
compute_agg_metrics(data_hybrid_bm_bge_with_reranker_with_metrics,
        is_simple_generation_metrics = True,
        simple_aggregation_generation_columns = ['language','answer_type'],
        simple_generation_metrics_columns=['bertscore_f1'],
        is_retriever_metrics=True,
        aggregation_retriever_columns=['language','answer_type'], 
        retriever_metrics_columns=retriever_metrics_columns,
        is_abstention_metrics = True,
        aggregation_abstention_columns =['language','answer_type'],
)

Simple generation:
           bertscore_f1
language              
en            0.836425
ru            0.836999 

Simple generation:
               bertscore_f1
answer_type               
abstention        0.788111
descriptive       0.831435
list              0.819937
short_entity      0.842566 

Retriever metrics:
           top_1_recall  top_3_recall  top_5_recall  top_1_precision  \
language                                                              
en            0.362700      0.515960      0.570120         0.700000   
ru            0.275235      0.458039      0.517882         0.490196   

          top_3_precision  top_5_precision  top_1_hit  top_3_hit  top_5_hit  \
language                                                                      
en               0.379920         0.276000   0.700000   0.780000   0.800000   
ru               0.307118         0.223529   0.490196   0.647059   0.686275   

          top_1_nDCG  top_3_nDCG  top_5_nDCG  context_precision  
language      

In [18]:

data_bge_with_reranker_with_metrics,result_bge_with_reranker=complete_eval_pipline(is_oracle=False,
                        is_generate_answers=True,
                        path_to_eval_set_with_chunks='/kaggle/input/datasets/vorange/labeled-eval-set-rag-system-anime/eval_set_chunks_sampled_labeled.csv',
                        retriever_model='bge-m3',
                        is_hybrid=False,
                        use_reranker=True,
                        top_k_for_reranker=5,
                        is_generation_metrics=True,
                        is_simple_generation_metrics=True,
                        is_time_metrics = True,
                        list_columns_time =['generation_time','e2e_latency'],
                        is_abstention_metrics =True,
                        queries_column='question',
                        answers_column='llm_answer',
                        chunks_column='chunks_from_retrieval',
                        relevant_chunks_column = 'llm_relevant_chunks_text',
                        ground_truth_column='ground_truth_text',
                        retriever_type='faiss',
                        faiss_index=faiss_index_bge,
                        top_k_chunks=20,
                        is_alone_retriever=True,
                        generation_source='local',
                        tokenizer=tokenizer,
                        local_generation_model=model,
                        name_evaluation_model='llama-3.1-8b-instant',
                        threshold = None,
                        is_retrieval_metrics=True,
                        top_k_recall=[1,3,5],
                        top_k_precision=[1,3,5],
                        top_k_hit=[1,3,5],
                        top_k_nDCG=[1,3,5],
                        is_context_precision=True

                                                      )

  0%|          | 0/101 [00:00<?, ?it/s]

/tmp/ipykernel_55/1275728909.py:122: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  return LangchainLLMWrapper(groq_llm)
/tmp/ipykernel_55/1275728909.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), per_group)))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 5.77 seconds, 17.51 sentences/sec
Exact Match: 0.000
BERTScore F1: 0.838
BERTScore Precision: 0.844
BERTScore Recall: 0.834
generation_time mean: 9.714734539831726
generation_time median: 7.327403789000073
generation_time p95 : 25.727539182999863
e2e_latency mean: 10.366886666920799
e2e_latency median: 7.977611952999723
e2e_latency p95 : 26.40217729699998
top_1_recall: 0.3193564356435643
top_3_recall: 0.5158415841584157
top_5_recall: 0.5674554455445544
top_1_precision: 0.5841584158415841
top_3_precision: 0.3662475247524753
top_5_precision: 0.2594059405940594
top_1_hit: 0.5841584158415841
top_3_hit: 0.7326732673267327
top_5_hit: 0.7722772277227723
top_1_nDCG: 0.5841584158415841
top_3_nDCG: 0.5712970297029704
top_5_nDCG: 0.5637029702970298
context_precision: 0.6413762376237624
Точность правильных отказов: 55.6%
Точность ложных отказов: 6.5%


In [21]:
data_bge_with_reranker_with_metrics.to_csv('data_bge_with_reranker_with_metrics.csv')